In [ ]:
from pathlib import Path

import pandas as pd
import yaml


repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / ".git").exists()
)

experiment_paths = {
    "LoRA - BUSI": "configs/experiment/sam3_lora_busi_full.yaml",
    "Full - BUSI": "configs/experiment/sam3_full_busi_full.yaml",
    "LoRA - Kvasir": "configs/experiment/sam3_lora_kvasir_full.yaml",
    "Full - Kvasir": "configs/experiment/sam3_full_kvasir_full.yaml",
}

experiment_rows = []

for experiment, relative_path in experiment_paths.items():
    config_path = repository_root / relative_path

    with open(config_path, encoding="utf-8") as config_file:
        config = yaml.safe_load(config_file)

    experiment_rows.append(
        {
            "experiment": experiment,
            "dataset": config["data"]["name"],
            "mode": config["model"]["finetune_mode"],
            "training_images": (
                "full"
                if config["data"]["n_train"] is None
                else config["data"]["n_train"]
            ),
            "batch_size": config["data"]["batch_size"],
            "augmentation": config["data"]["aug_preset"],
            "epochs": config["train"]["epochs"],
            "backbone_lr": config["train"]["backbone_lr"],
            "head_lr": config["train"]["head_lr"],
        }
    )

experiment_summary = pd.DataFrame(experiment_rows)
experiment_summary

,experiment,dataset,mode,training_images,batch_size,augmentation,epochs,backbone_lr,head_lr
0,LoRA — BUSI,busi,lora,full,1,medseg_default,20,0.00010,0.0001
1,Full — BUSI,busi,full,full,1,medseg_default,20,0.00001,0.0001
2,LoRA — Kvasir,kvasir,lora,full,1,medseg_default,20,0.00010,0.0001
3,Full — Kvasir,kvasir,full,full,1,medseg_default,20,0.00001,0.0001


In [2]:

import ast
import re


smoke_log_path = (
    repository_root / "slurm_outs" / "slurm_5542448.out"
)
smoke_log = smoke_log_path.read_text(
    encoding="utf-8",
    errors="replace",
)

splits = ast.literal_eval(
    re.search(r"splits: (\{.+?\})", smoke_log).group(1)
)
parameter_match = re.search(
    r"parameters: ([\d,]+) trainable / ([\d,]+) total "
    r"\(([\d.]+)%\)",
    smoke_log,
)
epoch_matches = re.finditer(
    r"epoch (\d+): "
    r"train_loss=([\d.]+)\s+"
    r"val_dice=([\d.]+)\s+"
    r"val_nsd=([\d.]+)",
    smoke_log,
)

smoke_summary = pd.Series(
    {
        "job_id": re.search(
            r"Job ID:\s+(\d+)", smoke_log
        ).group(1),
        "gpu": (
            "NVIDIA H100 80 GB"
            if "NVIDIA H100 80GB" in smoke_log
            else "not detected"
        ),
        "train_images": splits["train"],
        "validation_images": splits["val"],
        "test_images": splits["test"],
        "trainable_parameters": parameter_match.group(1),
        "total_parameters": parameter_match.group(2),
        "trainable_percent": float(parameter_match.group(3)),
        "peak_vram_gb": float(
            re.search(
                r"peak_vram=([\d.]+)GB",
                smoke_log,
            ).group(1)
        ),
        "walltime": re.search(
            r"walltime=([\d:]+)",
            smoke_log,
        ).group(1),
        "completed": (
            "done" in smoke_log
            and "Begin Slurm Epilog" in smoke_log
        ),
    },
    name="value",
)

smoke_epochs = pd.DataFrame(
    [
        {
            "epoch": int(match.group(1)),
            "train_loss": float(match.group(2)),
            "val_dice": float(match.group(3)),
            "val_nsd": float(match.group(4)),
        }
        for match in epoch_matches
    ]
)

display(smoke_summary.to_frame())
display(smoke_epochs)

,value
job_id,5542448
gpu,NVIDIA H100 80 GB
train_images,8
validation_images,64
test_images,131
trainable_parameters,"454,581,633"
total_parameters,"454,581,633"
trainable_percent,100.0
peak_vram_gb,9.4
walltime,00:02:58


,epoch,train_loss,val_dice,val_nsd
0,1,1.2893,0.5033,0.0567
1,2,0.7517,0.5939,0.0735


In [3]:
H100_MEMORY_GB = 80.0
EXPECTED_TRAINABLE_PERCENT = 100.0

first_epoch = smoke_epochs.iloc[0]
last_epoch = smoke_epochs.iloc[-1]

validation_checks = pd.Series(
    {
        "job_completed": bool(smoke_summary["completed"]),
        "all_parameters_trainable": (
            float(smoke_summary["trainable_percent"])
            == EXPECTED_TRAINABLE_PERCENT
        ),
        "training_loss_decreased": (
            last_epoch["train_loss"]
            < first_epoch["train_loss"]
        ),
        "validation_dice_improved": (
            last_epoch["val_dice"]
            > first_epoch["val_dice"]
        ),
        "fits_in_h100_memory": (
            float(smoke_summary["peak_vram_gb"])
            < H100_MEMORY_GB
        ),
    },
    name="passed",
)

display(validation_checks.to_frame())

assert validation_checks.all(), "Smoke validation failed"

loss_reduction = (
    first_epoch["train_loss"]
    - last_epoch["train_loss"]
)
dice_improvement = (
    last_epoch["val_dice"]
    - first_epoch["val_dice"]
)

print(f"Loss reduction: {loss_reduction:.4f}")
print(f"Validation Dice improvement: {dice_improvement:.4f}")
print("Decision: proceed with full BUSI and Kvasir runs.")

,passed
job_completed,True
all_parameters_trainable,True
training_loss_decreased,True
validation_dice_improved,True
fits_in_h100_memory,True


Loss reduction: 0.5376
Validation Dice improvement: 0.0906
Decision: proceed with full BUSI and Kvasir runs.
